#### MARIA Gradio Web App — based on gradio_sql_agent_v9.ipynb file (v1.6)

This script wraps the **AIS SQL Agent v1.6** (LangGraph + Postgres/PostGIS) in a **Gradio-based web chat application** so users can query the AIS database conversationally through a browser. It applies all agent logic from v1.6, including the dual-tool architecture (`execute_sql` + `generate_csv`), Pandas-based execution, XML-structured schema prompt, and the full validate → execute → heal guardrail loop.

##### What This File Adds

- **CSV Download Button (`gr.DownloadButton`):** A dedicated download button is rendered below the chatbot. It is hidden by default and becomes visible and populated only when a `generate_csv` tool call successfully produces a file. It resets to hidden when the user clears the chat.
- **Direct State-Based File Path Extraction:** The `respond()` function reads `csv_file_path` directly from the LangGraph state result (`result.get('csv_file_path')`). No regex or string parsing is used — the path is written by the executor node and read cleanly at the UI layer.

---

##### Key Upgrades vs Previous Gradio Version

- **`generate_csv` Tool Returns a Tuple:** Unlike the previous version (which returned a plain string with an embedded file path), `generate_csv` now returns a `tuple`: `(result_str, filepath)`. The executor node unpacks this directly — no regex needed — and stores the filepath in `AgentState.csv_file_path`.
- **`csv_file_path` State Field (NEW):** A new field is added to `AgentState` to reliably track the generated CSV file path across nodes. It is reset to `""` at the start of every new tool call in `agent_node` to prevent stale paths from a previous export being reused.
- **`execute_sql` Returns Row Count (NEW):** The `execute_sql` tool now prepends a `[Total N rows retrieved]` header to its output string, giving the LLM additional context for richer synthesis responses.
- **`gr.DownloadButton` Replaces File Component:** The previous version had no file download UI element. This version adds a `gr.DownloadButton` with `visible=False` as default, wired into the `respond()` function's output and the `clear_btn` reset handler (`on_clear()`).
- **Fixer Node Tool-Name Awareness:** The `fixer_node` now reads `current_tool_name` from state and passes it as `{tool_name}` into its debugger prompt, instructing the LLM to rewrite the broken query using the same tool the agent originally selected.
- **`clear_btn` Reset Handler:** A dedicated `on_clear()` function is bound to the clear button's `.click()` event, which resets the `gr.DownloadButton` to `visible=False` and `value=None` after the chat is cleared.

---

##### Execution Flow (UI → Agent → UI)

0. **Startup:** Load env vars → create SQLAlchemy engine → build `SQLDatabase` wrapper → define `execute_sql` and `generate_csv` tools → build LangGraph workflow → compile with `MemorySaver` → ensure `exports/` directory exists.
1. **User Message:** User types a question and presses Enter or clicks Submit.
2. **Session Bind:** `respond()` derives `session_id = request.session_hash` and passes it as `configurable.thread_id`.
3. **Agent Run:** LangGraph executes the v1.6 ReAct loop:
   - `agent_node` (selects `execute_sql` or `generate_csv`) → `validator_node` → `executor_node` → optional `fixer_node` retries → back to `agent_node` for synthesis.
4. **File Path Extraction:** After the graph completes, `respond()` reads `result.get('csv_file_path')` directly from the returned state.
5. **UI Update:**
   - The final agent message is appended to the Gradio chatbot as the assistant reply.
   - If a CSV was generated: `gr.DownloadButton` is set to `visible=True` with the file path as its value.
   - If no CSV was generated: `gr.DownloadButton` remains `visible=False`.
6. **Clear / Repeat:** The user can continue the conversation (state persists per session via `MemorySaver`), download a generated CSV, or clear the UI to reset both the chat history and the download button.

---


In [2]:
# --- TOP OF FILE IMPORTS ---
import os
import gradio as gr # <--- ADD THIS
from typing import TypedDict, Literal, Annotated
from langchain_openai import ChatOpenAI
from langchain_community.utilities import SQLDatabase
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph.message import add_messages
from langchain_core.messages import trim_messages

import pandas as pd
import uuid
from datetime import datetime
from sqlalchemy import create_engine

from dotenv import load_dotenv
load_dotenv()


# --- PHASE 1: DATABASE CONNECTIVITY ---

# --- DATABASE SETUP ---

def get_engine():
    db_user = os.getenv("DB_USER")
    db_password = os.getenv("DB_PASSWORD")
    db_host = os.getenv("DB_HOST")
    db_port = os.getenv("DB_PORT")
    db_name = os.getenv("DB_NAME")
    
    db_uri = f"postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"
    return create_engine(db_uri)

# Create the shared engine
engine = get_engine()

# Create the LangChain SQL wrapper using the same engine
db = SQLDatabase(
    engine,
    include_tables=['ais_data'],
    sample_rows_in_table_info=2
)

# --- PHASE 2: TOOL DEFINITION ---

@tool
def execute_sql(query: str) -> str:
    """
    Executes a PostgreSQL query against the 'ais_data' table.
    Returns a formatted text table with headers for analysis.
    Use this for answering general questions and statistics related AIS data columns and vessels.
    """
    try:
        # We use the engine directly with Pandas for better formatting
        df = pd.read_sql(query, engine)
        
        if df.empty:
            return "" # This makes len(result) == 0
        
        # Convert to string with headers, but no index numbers for cleaner context
        
        result_str = df.to_string(index=False)
        
        row_count = len(df)
        
        #ToolMessage also includes the row_count, to have extra contents while responding to the user.
        return f"[Total {row_count} rows retrieved]\n\nThe query result:\n{result_str}"
    
    except Exception as e:
        return f"ERROR: {str(e)}"
      

@tool
def generate_csv(query: str) -> tuple:
    """
    Executes a PostgreSQL query and exports the results as a CSV file.
    The file download is handled automatically by the application — do NOT 
    mention file paths, sandbox links, or download URLs in your response.
    Just confirm the export was successful and summarize the data.
    """
    try:
        df = pd.read_sql(query, engine)

        if df.empty:
            return ("", None)

        # --- Save the CSV ---
        os.makedirs("exports", exist_ok=True)
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"ais_export_{timestamp}_{uuid.uuid4().hex[:6]}.csv"
        filepath = os.path.join("exports", filename)
        df.to_csv(filepath, index=False)

        # --- Return string for LLM reasoning (same pattern as execute_sql) ---
        result_str = df.to_string(index=False)
        
        row_count = len(df)
    
        #Changing the tool return to a tuple (str, str).
        # First str element is the query result that LLM will receive in it's ToolMessage
        # Second str element is the filepath value, which the gradio will use the create the downloadable file
        return (f"CSV export successful. Total {row_count} rows exported.\n\nThe query result:\n{result_str}", filepath)

    except Exception as e:
        return (f"ERROR: {str(e)}", None)


# --- PHASE 3: STATE MANAGEMENT ---

class AgentState(TypedDict, total=False):
    messages: Annotated[list, add_messages]
    schema_context: str
    sql_query: str
    current_tool_call_id: str
    current_tool_name: str        # NEW: "execute_sql" or "generate_csv"
    validation_status: str
    critique: str
    retry_count: int
    csv_file_path: str   # NEW: Reliably track the generated file    

# --- PHASE 4: NODE DEVELOPMENT ---

MODEL = "openai/gpt-oss-120b"
BASE_URL = "https://api.groq.com/openai/v1"

llm = ChatOpenAI(
    model=MODEL,
    base_url=BASE_URL,
    api_key=os.environ["API_KEY"],
    temperature=0.3,
)

AIS_SCHEMA_INFO = """
<database>
  <table>ais_data</table>
</database>

<schema>
  <column_definitions>
    - mmsi (text): Unique vessel identifier
    - basedatetime (timestamp): Time of position report. Format: 'YYYY-MM-DDTHH:MM:SS'
    - lat (float): Latitude
    - lon (float): Longitude
    - sog (float): Speed over ground in knots
    - cog (float): Course over ground in degrees
    - heading (float): True heading in degrees (0 to 359). Value 511 means 'Not Available'
    - vesselname (text): Name of the ship. ALWAYS uppercase
    - imo (text): IMO number. ALWAYS starts with 'IMO'
    - callsign (text): Radio call sign
    - vesseltype (integer): Numeric ITU-R M.1371 code for vessel category
    - status (integer): Numeric ITU-R M.1371 code for navigation status
    - length (float): Vessel length in meters
    - width (float): Vessel width in meters
    - draft (float): Vessel draft in meters
    - cargo (integer): Numeric code for cargo type
    - transceiverclass (text): AIS class. Exactly 'A' or 'B'
    - geometry (geometry): PostGIS Point column. See geometry rules below
  </column_definitions>

  <status_codes>
    0=Moving/Under way, 1=At anchor, 2=Not under command, 3=Restricted maneuverability,
    4=Constrained by draught, 5=Moored/Docked, 6=Aground, 7=Fishing, 8=Sailing,
    11=Towing astern, 12=Pushing ahead/towing alongside, 14=Search and Rescue active, 15=Undefined
  </status_codes>

  <vessel_type_codes>
    30=Fishing, 31=Towing, 32=Large Towing, 33=Dredging/Underwater ops, 34=Diving ops,
    35=Military ops, 36=Sailing, 37=Pleasure Craft, 40-49=High-Speed Craft (HSC),
    50=Pilot Vessel, 51=Search and Rescue, 52=Tugs, 53=Port Tenders, 54=Anti-pollution,
    55=Law Enforcement, 58=Medical, 60-69=Passenger Ships, 70-79=Cargo Ships, 80-89=Tankers
  </vessel_type_codes>
</schema>

<critical_rules>
  <rule id="1" name="Type and Status Mapping">
    When users ask for specific types of vessels or statuses, map their natural language to the integer codes.
    - Example: 'moving cargo ships'            -> vesseltype BETWEEN 70 AND 79 AND status = 0
    - Example: 'parked or docked tankers'      -> vesseltype BETWEEN 80 AND 89 AND status IN (1, 5)
    - Example: 'ships in distress or broken down' -> status IN (2, 6, 14)
  </rule>

  <rule id="2" name="Heading Nulls">
    If calculating averages, minimums, or maximums for heading, OR if filtering for valid headings, you MUST exclude 511 (e.g., `heading != 511`)
  </rule>

  <rule id="3" name="Current State Rule — Time-Series Deduplication">
    This is a time-series database. A single ship (mmsi) has many historical rows.
    For ANY query asking about "current", "present", "now", or general fleet-wide state
    (e.g., "Find all passenger ships", "How many tankers", "Show moving tugs"),
    you MUST deduplicate to the most recent ping per vessel.
    - ALWAYS use a CTE (Common Table Expression) with `DISTINCT ON (mmsi)` and `ORDER BY mmsi, basedatetime DESC`.
    - Apply your `WHERE` filters (vesseltype, status, etc.) inside the CTE to optimize performance.
    - Query your final results (SELECT columns, COUNT, AVG) from this CTE.
  </rule>

  <rule id="4" name="Vessel Name Lookup">
    - ALWAYS use exact equality with uppercase: WHERE vesselname = 'VESSEL NAME'
    - NEVER use ILIKE or partial matching unless the user explicitly requests fuzzy/partial search
  </rule>

  <rule id="5" name="Geometry Column">
    THE ONE ABSOLUTE RULE: The raw geometry column value must NEVER appear in the
    final SELECT output. Everything else follows from this.

    BANNED in SELECT — these will crash pandas/SQLAlchemy:
    - SELECT geometry                ← raw binary PostGIS type
    - SELECT ST_AsText(geometry)     ← SQLAlchemy still resolves the source column type
    - SELECT *                       ← pulls raw geometry along with other columns

    For location output, ALWAYS use lat and lon columns

    ALLOWED — geometry as intermediate input to a spatial function chain
    (These are safe because the final value returned to pandas is TEXT or FLOAT, not geometry):
    Example: - SELECT ST_AsText(ST_Project(geometry::geography, dist, az))       → returns TEXT
    Example: - SELECT ST_Distance(geometry::geography, ST_MakePoint(lon, lat)::geography) → returns FLOAT

    ALLOWED — geometry inside WHERE / JOIN / CTE filter clauses
    (geometry value is consumed by the function and never enters the result set):
    - WHERE ST_DWithin(geometry::geography, ST_MakePoint(lon, lat)::geography, meters)
    - WHERE ST_Within(geometry, ST_MakeEnvelope(xmin, ymin, xmax, ymax, 4326))

  </rule>

  <rule id="6" name="PREDICTIVE ANALYTICS & MATH RULES">
    If a user asks where a ship WILL BE in a future time or hour (Projection), you MUST use PostGIS `ST_Project`.
    - Distance math: `sog` is in knots. You must multiply `sog` by 0.514444 to get meters per second. Multiply that by the requested time in seconds.
    - Azimuth math: `cog` is in degrees. You must convert it using `radians(cog)`.
    - Always use the most recent ping by adding `ORDER BY basedatetime DESC LIMIT 1`.
  </rule>

  <rule id="7" name="Result Size Guard">
    Unless the user requests an aggregation (COUNT, AVG, etc.),
    always append LIMIT 10 to prevent massive data pulls.
    EXCEPTION: If the user uses the word "all" or specifies a number of records, omit the LIMIT clause entirely.
  </rule>
</critical_rules>

<examples>
  <example>
    <q>How many Class A tankers are currently moored?</q>
    <sql>
      WITH latest AS (
        SELECT DISTINCT ON (mmsi) mmsi
        FROM ais_data
        WHERE transceiverclass = 'A' AND vesseltype BETWEEN 80 AND 89 AND status = 5
        ORDER BY mmsi, basedatetime DESC
      )
      SELECT COUNT(*) FROM latest;
    </sql>
  </example>

  <example>
    <q>Find the current speed and heading of the ship named LEICESTER.</q>
    <sql>
      SELECT sog, heading
      FROM ais_data
      WHERE vesselname = 'LEICESTER'
      ORDER BY basedatetime DESC LIMIT 1;
    </sql>
  </example>

  <example>
    <q>Show me the names of 5 military ships that are currently moving.</q>
    <sql>
      WITH latest AS (
        SELECT DISTINCT ON (mmsi) mmsi, vesselname
        FROM ais_data
        WHERE vesseltype = 35 AND status = 0
        ORDER BY mmsi, basedatetime DESC
      )
      SELECT vesselname FROM latest LIMIT 5;
    </sql>
  </example>

  <example>
    <q>What is the average heading of moving passenger ships?</q>
    <sql>
      WITH latest AS (
        SELECT DISTINCT ON (mmsi) mmsi, heading
        FROM ais_data
        WHERE vesseltype BETWEEN 60 AND 69 AND status = 0 AND heading != 511
        ORDER BY mmsi, basedatetime DESC
      )
      SELECT AVG(heading) AS avg_heading FROM latest;
    </sql>
  </example>

  <example>
    <q>Where will the ship MISS CHRISTY be in 2 hours?</q>
    <sql>
      SELECT ST_AsText(
        ST_Project(geometry::geography, (sog * 0.514444) * (2 * 3600), radians(cog))
      ) AS predicted_location
      FROM ais_data
      WHERE vesselname = 'MISS CHRISTY'
      ORDER BY basedatetime DESC LIMIT 1;
    </sql>
  </example>

  <example>
    <q>How far is the ship ANTHEM OF THE SEAS from coordinates (25.0, -80.0)?</q>
    <sql>
      SELECT ST_Distance(
        geometry::geography,
        ST_MakePoint(-80.0, 25.0)::geography
      ) AS distance_meters
      FROM ais_data
      WHERE vesselname = 'ANTHEM OF THE SEAS'
      ORDER BY basedatetime DESC LIMIT 1;
    </sql>
  </example>
</examples>
"""



VALIDATOR_SCHEMA_INFO = """
Table: ais_data
Columns: mmsi (text), basedatetime (timestamp), lat (float), lon (float), sog (float), cog (float), heading (float, ignore 511), vesselname (text), imo (text), callsign (text), vesseltype (int), status (int), length (float), width (float), draft (float), cargo (int), transceiverclass (text, 'A' or 'B'), geometry (PostGIS point).
Valid status codes: 0, 1, 2, 3, 4, 5, 6, 7, 8, 11, 12, 14, 15.
Valid vesseltype codes: 30-37, 40-55, 58, 60-69 (Passenger), 70-79 (Cargo), 80-89 (Tankers).

SAFE FUNCTIONS ALLOWED:
- Standard Aggregates: COUNT, AVG, MIN, MAX
- PostGIS Spatial & Projection: ST_Project, ST_AsText, ST_Distance, ST_DWithin, ST_MakePoint, ST_MakeEnvelope
- Math/Casting: radians(), ::geography, basic multiplication/division.
"""

def agent_node(state: AgentState):
    """Node A: The Brain (Reasoning & Synthesis)"""
    print("\n--- AGENT THINKING ---")
    
    # --- MEMORY PRUNING (SLIDING WINDOW) ---
    # Keeps the last 10 messages in the array.
    # allow_partial=False ensures we never accidentally separate an AIMessage tool-call 
    # from its corresponding ToolMessage, preventing API crash errors.
    pruned_messages = trim_messages(
        state['messages'],
        max_tokens=25, 
        token_counter=len,
        strategy="last",
        allow_partial=False,
        start_on="human" # <--- NEW: Forces the array to always start with a user query
    )
    
    
    # --- MAX RETRY SHORT-CIRCUIT: After the agent hit the maximum fixing retries ---
    if state.get('retry_count', 0) > 3:
        print("Decision: Max retries detected, Stopping the process. ")
        apology_prompt = ChatPromptTemplate.from_messages([
            ("system", """You are a helpful maritime AI assistant.
            A database query has failed after multiple attempts. 
            Apologize clearly, explain you could not retrieve the data,
            and suggest the user to double check on the information given in the question or give more information.
            Do NOT attempt any further queries."""),
            MessagesPlaceholder(variable_name="messages")
        ])
        chain = apology_prompt | llm  # No tools bound
        response = chain.invoke({"messages": pruned_messages})
        return {"messages": [response], "schema_context": AIS_SCHEMA_INFO}
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", """You are an expert maritime AI assistant and PostGIS database analyst. 
        You have access to a PostgreSQL database containing AIS ship telemetry data.
        
        Database Schema & Translation Rules:
        {schema}
        
        CORE INSTRUCTIONS:
        
        1. WHICH TOOL TO USE (`execute_sql` OR `generate_csv`)
        Both tools are almost same, except the `generate_csv` tool has extra capability to export, download or save the query results in a csv file.
        - Use `execute_sql` for general data questions, statistics, vessel details, locations and analysis.
        - Use `generate_csv` ONLY when the user explicitly requests a report file, CSV, export, or download.
        
        2. WHEN TO USE THE DATABASE (TOOL USE): 
        If the user asks for specific data, statistics, vessel details, or spatial locations, you MUST use the `execute_sql` tool to generate and run a PostgreSQL query. If the user also requests a file, CSV, export, or download, you MUST use `generate_csv` tool instead.
        - ALWAYS use 'ais_data' as the table name. Never hallucinate column names. Strictly adhere to the columns and definitions in the schema provided.
        - When a question has multiple related parts (e.g., "list X and also compute Y of X"), you MUST first attempt to answer ALL parts in a SINGLE SQL query before considering separate calls. Use window functions or multiple SELECT columns.
        - If the user asks a complex question requiring multiple pieces of UNRELATED information, you can use the `execute_sql` tool multiple times in sequence to gather all the data you need.

        3. WHEN TO CHAT (NO TOOL USE): 
        If the user says hello, asks general maritime knowledge, or explanation of a value from the AIS database (e.g., "What does AIS stand for?" or "What does the Vessel Status mean?"), or asks an unrelated question, DO NOT use the tool. Provide a helpful, natural language response directly.
        
        SYNTHESIS INSTRUCTION:
        1. If you had to use the DATABASE (TOOL USE), ONLY synthesize a final response AFTER you have successfully retrieved all required information using the `execute_sql` tool or the `generate_csv` tool.
        2. If you didn't have to use any of the DATABASE TOOLS, you may answer naturally without using any tools.
        3. CRITICAL — CSV EXPORTS: When the tool confirms a CSV export was successful, DO NOT generate any file path, download link, or URL of any kind. 
          The download button is handled automatically by the UI. Simply confirm the export, state the row count, and summarize the data preview.
        
        """),
        MessagesPlaceholder(variable_name="messages")
    ])
    
    llm_with_tools = llm.bind_tools([execute_sql, generate_csv])  # Add generate_csv
    chain = prompt | llm_with_tools
    
    # We pass 'pruned_messages' to the LLM instead of the full 'state['messages']'
    response = chain.invoke({
        "schema": AIS_SCHEMA_INFO, 
        "messages": pruned_messages 
    })
    
    # Update state with the agent's new message
    state_update = {"messages": [response], "schema_context": AIS_SCHEMA_INFO}
    
    # Did the agent call the tool?
    if response.tool_calls:
        tool_call = response.tool_calls[0]
        state_update["sql_query"] = tool_call['args']['query']
        state_update["current_tool_call_id"] = tool_call['id']
        state_update["current_tool_name"] = tool_call['name']   # NEW
        state_update["retry_count"] = 0 # Reset retries for a new query
        state_update["csv_file_path"] = ""  # Reset stale path on each new tool call
        print(f"Decision: Tool call requested, selected Tool -> {state_update['current_tool_name']}. \nQuery:\n {state_update['sql_query']}")
    else:
        print("Decision: Gathered enough context. Providing final answer.")
        
    return state_update

def validator_node(state: AgentState):
    """Node B: The Critic"""
    print("--- VALIDATING SQL ---")
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", """You are a strict PostgreSQL Security Validator.
        Review the following query for the `ais_data` table against this schema:
        {validation_schema}
        
        CRITICAL CHECKS:
        1. READ-ONLY: Reject if it contains DROP, DELETE, INSERT, UPDATE, ALTER.
        2. HALLUCINATIONS: Reject if it queries a column name NOT listed in the schema.
        3. DATA TYPES: Reject if `status` or `vesseltype` are queried as text strings instead of their valid integers.
        
        Output format: Return exactly "VALID" if safe. If invalid, return a concise explanation of the exact error."""),
        ("human", "Query to check: {query}")
    ])
    
    response = (prompt | llm).invoke({
        "query": state['sql_query'],
        "validation_schema": VALIDATOR_SCHEMA_INFO 
    })
    
    status = "VALID" if "VALID" in response.content.upper() else "INVALID"
    if status == "INVALID":
        print(f"Validation Failed: {response.content}")
    
    return {"validation_status": status, "critique": response.content if status == "INVALID" else ""}

def executor_node(state: AgentState):
    """Node C: The Executor"""
    print("--- EXECUTING SQL---")
    
    tool_name = state.get("current_tool_name", "execute_sql")
    
    try:
        if tool_name == "generate_csv":
            # Unpack the tuple directly — no regex needed
            result, saved_filepath = generate_csv.invoke({"query": state['sql_query']})
            print(f"DEBUG — generate_csv returned filepath: {saved_filepath}")
        else:
            result = execute_sql.invoke({"query": state['sql_query']})
            saved_filepath = None
            
        if isinstance(result, str) and result.startswith("ERROR:"):
            print("--- DATABASE ERROR CAUGHT ---")
            
            #------ Extra Print
            print("\n The error is--- \n", result, "\n")
            return {"critique": f"Database Execution Error: {result}", "validation_status": "INVALID"}

        if result == "":
            result = "No results found. The requested vessel or entity does not exist in the database. Do NOT retry — synthesize a 'not found' response and clearly apologise for not being able to help or do what was asked."
            return {
                "messages": [ToolMessage(content=result, tool_call_id=state['current_tool_call_id'])],
                "critique": ""
            }
        
        print(f"Execution successful. Result length: {len(result)}")
        print("\n The query result is--- \n", result, "\n")

        state_update = {
            "messages": [ToolMessage(content=str(result), tool_call_id=state['current_tool_call_id'])],
            "critique": ""
        }

        # Path comes directly from the tool — no regex, no guessing
        if saved_filepath and os.path.exists(saved_filepath):
            state_update["csv_file_path"] = saved_filepath
            print(f"CSV path stored in state: {saved_filepath}")

        return state_update    
    
    except Exception as e:
        print("--- CRITICAL SYSTEM ERROR CAUGHT ---")
        return {"critique": f"System Invocation Error: {str(e)}", "validation_status": "INVALID"}

def fixer_node(state: AgentState):
    """Node D: The Repairman (Invisible to the main Agent loop)"""
    print("--- FIXING SQL ---")
    current_retries = state.get('retry_count', 0)
    
    # Prevent infinite fixing loops. If we fail 3 times, send an error back to the Agent.
    if current_retries >= 3:
        print("--- MAX RETRIES REACHED. RETURNING ERROR TO AGENT ---")
        return {
            "messages": [ToolMessage(content="ERROR: Failed to execute query due to persistent database errors. Notify the user.", tool_call_id=state['current_tool_call_id'])],
            "retry_count": current_retries + 1
        }
    
    # Extract the user's original intent from the message history to give the fixer context
    user_intent = next((msg.content for msg in reversed(state['messages']) if isinstance(msg, HumanMessage)), "Unknown Context")

    original_tool = state.get("current_tool_name", "execute_sql")
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", """You are an Expert PostgreSQL Debugger. 
        Fix this failed query.
        Original Request: {question}
        Broken Query: {query}
        Critique: {critique}
        
        Schema Rules: {schema}
        Task: Rewrite the query permanently fixing the error. Use `{tool_name}` tool."""),
    ])
    
    llm_with_tools = llm.bind_tools([execute_sql, generate_csv], tool_choice="required")
    response = (prompt | llm_with_tools).invoke({
        "question": user_intent,
        "query": state['sql_query'],
        "critique": state['critique'],
        "schema": state['schema_context'],
        "tool_name": original_tool
    })
    
    fixed_sql = response.tool_calls[0]['args']['query']
    print(f"Proposed Fix: {fixed_sql}")
    
    return {"sql_query": fixed_sql, "retry_count": current_retries + 1}

# --- PHASE 5: GRAPH CONSTRUCTION & ROUTING ---

def route_agent_action(state: AgentState) -> Literal["validator", END]:
    """If the agent called a tool, go validate it. If it just output text, we are done."""
    last_message = state['messages'][-1]
    if hasattr(last_message, 'tool_calls') and last_message.tool_calls:
        return "validator"
    return END

def should_continue_validation(state: AgentState) -> Literal["executor", "fixer"]:
    return "executor" if state['validation_status'] == "VALID" else "fixer"

def should_continue_execution(state: AgentState) -> Literal["agent", "fixer"]:
    """If critique is empty, it means execution succeeded -> loop back to Agent."""
    return "agent" if not state.get("critique") else "fixer"

def should_continue_fixing(state: AgentState) -> Literal["validator", "agent"]:
    """If max retries hit, go back to Agent so it can synthesize an apology."""
    if state.get('retry_count', 0) > 3:
        return "agent" 
    return "validator"


workflow = StateGraph(AgentState)

workflow.add_node("agent", agent_node)
workflow.add_node("validator", validator_node)
workflow.add_node("executor", executor_node)
workflow.add_node("fixer", fixer_node)

workflow.add_edge(START, "agent")
workflow.add_conditional_edges("agent", route_agent_action)
workflow.add_conditional_edges("validator", should_continue_validation)
workflow.add_conditional_edges("executor", should_continue_execution)
workflow.add_conditional_edges("fixer", should_continue_fixing)


# ==========================================
# PHASE 6: COMPILATION & GRADIO BLOCKS UI
# ==========================================
import gradio as gr
import re
import os
from langchain_core.messages import HumanMessage

# Ensure memory and workflow compilation
memory = MemorySaver() 
app = workflow.compile(checkpointer=memory)

def respond(user_input, chat_history, request: gr.Request):
    """
    Handles the user input, queries the LangGraph agent, and extracts CSV file paths.
    """
    if not user_input.strip():
        return "", chat_history, gr.DownloadButton(value=None, visible=False)
        
    # Bind the Gradio session to the LangGraph thread to separate user sessions
    session_id = request.session_hash if request else "default_session"
    config = {"configurable": {"thread_id": session_id}}
    
    print(f"\n--- New Message from {session_id} ---")
    print(f"User: {user_input}")
    
    # 1. Append the user's message to the UI history
    chat_history.append({"role": "user", "content": user_input})
    
    file_path = None
    try:
        # Send the message to your LangGraph app
        result = app.invoke({"messages": [HumanMessage(content=user_input)]}, config=config)
        
        # Extract the final answer from MARIA
        ai_response = result['messages'][-1].content

        # No regex, no guessing — read directly from state
        file_path = result.get('csv_file_path') or None

    except Exception as e:
        ai_response = f"An error occurred while processing your request: {str(e)}"
        
    # 2. Append the AI's response to the UI history
    chat_history.append({"role": "assistant", "content": ai_response})
    
    # Show button only when a file exists, hide it otherwise
    visibility_update = gr.DownloadButton(value=file_path, visible=file_path is not None)
    return "", chat_history, visibility_update


# --- CUSTOM UI LAYOUT ---

custom_css = """
/* 1. Base text, lists, tables, and quotes */
.message-wrap p, 
.message-wrap li, 
.message-wrap td, 
.message-wrap th,
.message-wrap blockquote { 
    font-size: 14px !important; 
    line-height: 1.5 !important;
}

/* 2. Code blocks (inline and multi-line) */
.message-wrap code,
.message-wrap pre {
    font-size: 13px !important;
}

/* 3. Headers (Scaled down to match the 14px base text) */
.message-wrap h1 { font-size: 16px !important; margin-top: 8px !important; margin-bottom: 4px !important; }
.message-wrap h2 { font-size: 15px !important; margin-top: 8px !important; margin-bottom: 4px !important; }
.message-wrap h3, .message-wrap h4, .message-wrap h5, .message-wrap h6 { 
    font-size: 14px !important; font-weight: 600 !important; margin-top: 8px !important; margin-bottom: 4px !important; 
}

/* 4. Fix list spacing */
.message-wrap ul, .message-wrap ol {
    margin-top: 4px !important; margin-bottom: 4px !important; padding-left: 20px !important;
}

/* 5. Visual Separation for the File Component */
#file_download {
    margin-top: 10px !important;
    margin-bottom: 10px !important;
}


/* --- DOWNLOAD BUTTON: Target the inner element Gradio actually renders --- */
#file_download,
#file_download .wrap,
#file_download > .container,
#file_download a,
#file_download button,
div#file_download > * {
    background: linear-gradient(135deg, #1a7f4b, #22a863) !important;
    color: white !important;
    border: none !important;
    border-radius: 8px !important;
    font-weight: 600 !important;
    font-size: 14px !important;
    box-shadow: 0 2px 6px rgba(26, 127, 75, 0.4) !important;
    transition: background 0.2s ease, box-shadow 0.2s ease !important;
}

#file_download:hover,
#file_download a:hover,
#file_download button:hover {
    background: linear-gradient(135deg, #166038, #1a9955) !important;
    box-shadow: 0 4px 12px rgba(26, 127, 75, 0.5) !important;
}
"""

maria_theme = gr.themes.Soft(
    font=[gr.themes.GoogleFont("Inter"), "ui-sans-serif", "system-ui", "sans-serif"],
    font_mono=[gr.themes.GoogleFont("Fira Code"), "ui-monospace", "Consolas", "monospace"],
    text_size=gr.themes.sizes.text_sm 
)

with gr.Blocks(title="MARIA - AIS RAG Chatbot", css=custom_css) as demo:
    
    # Header Section
    gr.Image(
        value="MARIA_logo.png", 
        show_label=False,
        container=False,
        interactive=False,
        buttons=[],
        elem_id="logo",
        height=70
    )
    gr.Markdown(
        """
        <div style='text-align: center; line-height: 1;'>
            <h2 style='margin: 0 0;'>MARIA v2.0</h2>
            <p style='margin: 0; font-size: 14px;'>Maritime AIS Retrieval & Intelligence Assistant</p>
        </div>
        """
    )

    # Chat Interface Section
    chatbot = gr.Chatbot(label="MARIA", height=500)
    
    # VISUAL SEPARATION: File component for CSV downloads
    file_download = gr.DownloadButton(
        label="⬇  Download CSV Report", 
        visible=False,       # Hidden until a CSV is actually generated
        elem_id="file_download"
    )
    
    with gr.Row():
        message = gr.Textbox(placeholder="Type your question here...", show_label=False, scale=8)
        submit_btn = gr.Button("Submit", scale=1)

    clear_btn = gr.ClearButton([message, chatbot, file_download])
    
    def on_clear():
        """Reset the download button to hidden after clearing the chat."""
        return gr.DownloadButton(value=None, visible=False)

    clear_btn.click(fn=on_clear, inputs=None, outputs=[file_download])

    # Event Listeners - Updated to handle the File output
    submit_event = {
        "fn": respond, 
        "inputs": [message, chatbot], 
        "outputs": [message, chatbot, file_download]  # same output target
    }

    
    message.submit(**submit_event)
    submit_btn.click(**submit_event)

    # Footer Section
    gr.Markdown(
        """
        <div style='text-align: center; font-size:18px;'>
            <hr>
            <strong>IISE 2026</strong><br><br>
            Collaborators<br>
            <strong>Zaidur Rahman</strong> & <strong>Dr. Heather Nachtmann</strong><br>
            <img src="https://brand.uark.edu/_resources/images/UA_Logo.png" 
                 alt="University of Arkansas Logo" 
                 width="120" 
                 style="display:block; margin-left:auto; margin-right:auto; margin-top: 10px;">
        </div>
        """
    )

if __name__ == "__main__":
    # Ensure the directory for exports exists so the file system doesn't error out
    os.makedirs("exports", exist_ok=True)
    print("Launching MARIA Gradio Web App...")
    demo.launch(share=True, theme=maria_theme)

/Users/zaidur/anaconda3/envs/lang_env/lib/python3.10/site-packages/langchain_community/utilities/sql_database.py:159: SAWarning: Did not recognize type 'geometry' of column 'geometry'
  self._metadata.reflect(
/var/folders/8f/dbt9gbp93cbc6ns69dz8rq6c0000gn/T/ipykernel_76698/2700655019.py:702: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(title="MARIA - AIS RAG Chatbot", css=custom_css) as demo:


Launching MARIA Gradio Web App...
* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://4e902c2410f6a97566.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



--- New Message from 9rfehi81bem ---
User: Okay, I want a downloadable report of first 15 fishing vessels with their last known position.

--- AGENT THINKING ---
Decision: Tool call requested, selected Tool -> generate_csv. 
Query:
 WITH latest AS (
  SELECT DISTINCT ON (mmsi) mmsi, vesselname, lat, lon, basedatetime
  FROM ais_data
  WHERE vesseltype = 30
  ORDER BY mmsi, basedatetime DESC
)
SELECT mmsi, vesselname, lat, lon, basedatetime
FROM latest
ORDER BY vesselname
LIMIT 15;
--- VALIDATING SQL ---
--- EXECUTING SQL---
DEBUG — generate_csv returned filepath: exports/ais_export_20260302_014440_9e0cba.csv
Execution successful. Result length: 780

 The query result is--- 
 CSV export successful. Total 10 rows exported.

The query result:
     mmsi     vesselname      lat        lon        basedatetime
367643270 AMERICAN DREAM 30.88271  -87.98396 2024-01-05 23:59:15
338137000          ARICA 47.62801 -122.38211 2024-01-05 23:59:16
368132340   BOLD VENTURE 29.93929  -91.84605 2024-01-0

In [3]:
demo.close()

Closing server running on port: 7861
